In [1]:
from packages.transcripts import transcribe_with_timestamps
from packages.PII_detection import detect_pii_entities
from packages.mapTimeStamps import map_pii_to_timestamps, map_pii_to_timestamps_with_words

In [2]:
input_audio_path = r"D:\Coding\PII Evaluation\main server dataset\phase 3\4547022.wav"

In [3]:
transcript = transcribe_with_timestamps(input_audio_path)
print(transcript)

{'text': 'मेरा नाम दीपा है, गांव नगर नौशा, जिला नालंदा नमस्कार श्रोताओं, मेरी आवाज़ मेरी पहचान में आप लोगों का स्वागत है। हम मीटिंग में एक दीदी से मिलते हैं और उनसे पूछते हैं कि हमारे गांव का मुखिया कैसा होना चाहिए? हाँ तो दीदी आप बताइए आपका नाम क्या है? चार बच्चे है, एक चार। मेरा नाम संगीता वर्मा है। कमल जीव का जीविका समूह सर का मैं अध्यक्ष हूँ, मेरे गांव का मुखिया ऐसा हो की गांव का विकास में सहयोग करें, नल्ली गली पक्की कराये और महिला विकास के और ध्यान दें और शौचालय हर घर में और बिजली की योजना का लाभ कराएं। ऐसा मुखिया हो हमारे गरीब अमीर के दोनों को देख रेख करना है। बहुत बहुत धन्यवाद दीदी आप मुखिया के बारे में इतना जानकारी हमें दिए ', 'chunks': [{'text': 'मेरा', 'timestamp': (0.07, 0.47)}, {'text': 'नाम', 'timestamp': (0.47, 0.71)}, {'text': 'दीपा', 'timestamp': (0.71, 1.03)}, {'text': 'है', 'timestamp': (1.03, 1.23)}, {'text': 'गांव', 'timestamp': (1.23, 1.47)}, {'text': 'नगर', 'timestamp': (1.47, 1.71)}, {'text': 'नौशा', 'timestamp': (1.71, 2.03)}, {'text': 'जिला', 'timestamp': (2.03

In [4]:
asr_text = [transcript['text']]
pii_result = detect_pii_entities(asr_text)
pii_words = []
for doc in pii_result:
    print("Redacted Text: {}".format(doc.redacted_text))
    for entity in doc.entities:
        if entity.confidence_score > 0.7 and entity.category in  ['Person', 'Organization', 'PhoneNumber', 'Address', 'Location', 'PhoneNumber', 'BankAccountNumber']:
            print("Entity: {}".format(entity.text))
            pii_words.append(entity.text)
            print("\tCategory: {}".format(entity.category))
            print("\tConfidence Score: {}".format(entity.confidence_score))
            print("\tOffset: {}".format(entity.offset))
            print("\tLength: {}".format(entity.length))

Redacted Text: मेरा नाम **** है, गांव नगर नौशा, जिला नालंदा नमस्कार श्रोताओं, मेरी आवाज़ मेरी पहचान में आप लोगों का स्वागत है। हम मीटिंग में एक **** से मिलते हैं और उनसे पूछते हैं कि हमारे गांव का ****** कैसा होना चाहिए? हाँ तो **** आप बताइए आपका नाम क्या है? चार बच्चे है, एक चार। मेरा नाम ************ है। ******* का जीव******** सर का मैं ******* हूँ, मेरे गांव का ****** ऐसा हो की गांव का विकास में सहयोग करें, नल्ली गली पक्की कराये और महिला विकास के और ध्यान दें और शौचालय हर घर में और बिजली की योजना का लाभ कराएं। ऐसा ****** हो हमारे गरीब अमीर के दोनों को देख रेख करना है। बहुत बहुत धन्यवाद **** आप ****** के बारे में इतना जानकारी हमें दिए 
Entity: दीपा
	Category: Person
	Confidence Score: 0.99
	Offset: 9
	Length: 4
Entity: दीदी
	Category: Person
	Confidence Score: 0.93
	Offset: 213
	Length: 4
Entity: संगीता वर्मा
	Category: Person
	Confidence Score: 1.0
	Offset: 276
	Length: 12


In [4]:
pii_words = ['दीपा', 'दीदी', 'संगीता वर्मा']

In [5]:
pii_segments = map_pii_to_timestamps(pii_words, transcript['chunks'])
# pii_segments = map_pii_to_timestamps_with_words(pii_words, transcript['chunks'])
print(pii_segments)

[(0.71, 1.03), (7.43, 7.67), (11.75, 11.99), (41.11, 41.47), (16.47, 16.95), (16.95, 17.31)]


In [ ]:
import os
import json

# Define the folder path
folder_path = r"D:\Coding\PII Evaluation\Annotations output"

# Iterate over all files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        file_path = os.path.join(folder_path, filename)
        ground_truth=[]
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)

                print(f"--- Processing File: {filename} ---")
                
                # Extract and convert segments to seconds
                if "data" in data and "segments" in data["data"]:
                    segments = data["data"]["segments"]
                    print("Segments (in seconds):")
                    for segment in segments:
                        start_sec = segment["start"] / 1000
                        end_sec = segment["end"] / 1000
                        ground_truth.append((start_sec, end_sec))
                        print(f"  Start: {start_sec}s, End: {end_sec}s")

                # Fetch the file map for the WAV file
                if "file_map" in data:
                    wav_file_name = data["file_map"].get(data["files"]["audio"])
                    print(f"Mapped WAV file name: {wav_file_name}")

                print("\n")

        except json.JSONDecodeError:
            print(f"Error decoding JSON from {filename}. Skipping.")
        except KeyError:
            print(f"Missing expected keys in {filename}. Skipping.")

--- Processing File: DS-19768.DP-0bbad67112dd4f3d8ae0a2b96d14184a.TS-2025-08-08T05-58-10.671Z.json ---
Segments (in seconds):
  Start: 0.61s, End: 0.9s
  Start: 1.23s, End: 2.87s
  Start: 3.76s, End: 5.2s
  Start: 16.35s, End: 17.21s
  Start: 17.42s, End: 20.62s
Mapped WAV file name: 4547022.wav


--- Processing File: DS-19768.DP-189131f00f2040f19a65d792398ca3d2.TS-2025-08-08T05-58-10.671Z.json ---
Segments (in seconds):
  Start: 0.56s, End: 1.27s
  Start: 6.04s, End: 7.16s
  Start: 7.75s, End: 8.02s
  Start: 11.0s, End: 11.36s
  Start: 12.31s, End: 13.11s
  Start: 46.86s, End: 48.695s
  Start: 52.635s, End: 53.735s
Mapped WAV file name: 5373349.wav


--- Processing File: DS-19768.DP-24fa4d88295749bb9989b2f7542fc3d6.TS-2025-08-08T05-58-10.671Z.json ---
Segments (in seconds):
  Start: 1.33s, End: 1.66s
  Start: 3.07s, End: 3.62s
  Start: 7.49s, End: 8.35s
  Start: 18.57s, End: 19.22s
  Start: 24.42s, End: 25.13s
  Start: 27.11s, End: 29.04s
  Start: 89.78s, End: 91.65s
  Start: 273.54s,

In [6]:
file_path=r"D:\Coding\PII Evaluation\Annotations output\DS-19768.DP-0bbad67112dd4f3d8ae0a2b96d14184a.TS-2025-08-08T05-58-10.671Z.json"

In [11]:
import json

def get_ground_truth_from_json(file_path):
    """
    Reads a single JSON file and extracts the ground truth segments in seconds.
    
    Args:
        file_path (str): The full path to the JSON annotation file.
        
    Returns:
        list: A list of (start_sec, end_sec) tuples, or an empty list if
              the file is invalid or required data is missing.
    """
    ground_truth = []
    
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
            
            # Extract and convert segments to seconds
            if "data" in data and "segments" in data["data"]:
                for segment in data["data"]["segments"]:
                    start_sec = segment["start"] / 1000
                    end_sec = segment["end"] / 1000
                    ground_truth.append((start_sec, end_sec))
            
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
    except json.JSONDecodeError:
        print(f"Error decoding JSON from {file_path}. Skipping.")
    except KeyError:
        print(f"Missing expected keys in {file_path}. Skipping.")
        
    return ground_truth



# Get the ground_truth list
ground_truth = get_ground_truth_from_json(file_path)

print(f"Ground truth segments for the file: {ground_truth}")

Ground truth segments for the file: [(0.61, 0.9), (1.23, 2.87), (3.76, 5.2), (16.35, 17.21), (17.42, 20.62)]


In [8]:
sorted_ranges = sorted(pii_segments, key=lambda x: x)  # get the timestamps in a list

print(sorted_ranges)

[(0.71, 1.03), (7.43, 7.67), (11.75, 11.99), (16.47, 16.95), (16.95, 17.31), (41.11, 41.47)]


In [9]:
def calculate_overlap(range1, range2):
    """
    Calculate the overlap duration between two timestamp ranges.

    Parameters:
        range1 (tuple): (start1, end1) in seconds
        range2 (tuple): (start2, end2) in seconds
    
    Returns:
        float: Overlap duration in seconds. 0 if no overlap.
    """
    start1, end1 = range1
    start2, end2 = range2

    # Find the latest start time and earliest end time
    overlap_start = max(start1, start2)
    overlap_end = min(end1, end2)

    # If the intervals overlap, overlap_start < overlap_end
    if overlap_start < overlap_end:
        return overlap_end - overlap_start
    else:
        return 0


def evaluate_ranges_any_overlap(ground_truth, predictions):
    tp, fp, fn, tn = 0, 0, 0, 0
    matched_gt = set()
    matched_pred = set()
    
    for pred_idx, pred_range in enumerate(predictions):
        found_overlap = False
        for gt_idx, gt_range in enumerate(ground_truth):
            if calculate_overlap(pred_range, gt_range) > 0:
                if gt_idx not in matched_gt and pred_idx not in matched_pred:
                    tp += 1
                    matched_gt.add(gt_idx)
                    matched_pred.add(pred_idx)
                    found_overlap = True
                    break
        if not found_overlap:
            fp += 1

    fn = len(ground_truth) - len(matched_gt)

    # Calculate TN: count of prediction/ground_truth pairs that do not overlap and are not already matched
    total_pairs = len(ground_truth) * len(predictions)
    overlap_pairs = 0
    for pred_idx, pred_range in enumerate(predictions):
        for gt_idx, gt_range in enumerate(ground_truth):
            if calculate_overlap(pred_range, gt_range) > 0:
                overlap_pairs += 1
    tn = total_pairs - overlap_pairs

    return tp, fp, fn, tn

In [12]:
tp, fp, fn, tn= evaluate_ranges_any_overlap(ground_truth, sorted_ranges)

# Calculate standard metrics
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn)> 0 else 0

f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
f2_score = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0

print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Negatives: {tn}")
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1_score:.3f}")
print(f"F2 Score: {f2_score:.3f}")

True Positives: 2
False Positives: 4
False Negatives: 3
True Negatives: 27
Accuracy: 0.806
Precision: 0.333
Recall: 0.400
F1 Score: 0.364
F2 Score: 0.385
